# ASHRAE Great Energy Predictor III — IoT Data Cleaning, Feature Engineering & SQL Analytics

**Course:** ISTM 6217 - Internet of Things Management
**Dataset:** [ASHRAE - Great Energy Predictor III (Kaggle)](https://www.kaggle.com/competitions/ashrae-energy-prediction)
**Group Members:** Stacey Chen, Tatenda Kagande, Alana Lee, Safi Marghub, Nqobani Masayile

**Version 2:** cleans the meter data, aligns weather and meter time zones, compares meter types fairly, and adds the proposal's remaining analysis: energy use intensity, temperature effects, prediction models and anomaly detection (Sections 5.6–8).

<a id="toc"></a>
## Table of Contents
1. [Environment Setup & Library Imports](#1-env-setup)
2. [Dataset Download & Extraction](#2-data-download)
3. [Spark Session & Data Loading](#3-spark-loading)
4. [Data Cleaning & Feature Engineering](#4-cleaning-fe)
   * [4.1 Building Metadata Cleaning & Age Cohorts](#4-1-building-metadata)
   * [4.2 Meter Data Cleaning](#4-2-train-data)
   * [4.3 Weather Data Cleaning & Time/Rolling Features](#4-3-weather-data)
   * [4.4 Building Size Quartiles & Schema Verification](#4-4-building-features)
5. [Business Questions & Spark SQL Analysis](#5-business-questions)
   * [5.1 Daily Operational Cycles](#5-1-daily-cycles)
   * [5.2 Weekly Usage Drops](#5-2-weekly-drops)
   * [5.3 Seasonal Fuel Switching](#5-3-seasonal-switching)
   * [5.4 Temperature Thresholds](#5-4-temp-thresholds)
   * [5.5 Weather & Building Age Efficiency](#5-5-age-efficiency)
   * [5.6 Energy Use Intensity by Primary Use & Size](#5-6-eui)
   * [5.7 Temperature vs Usage](#5-7-temp-corr)
6. [Gold Feature Table & Prediction Models](#6-modeling)
   * [6.1 Gold Feature Table](#6-1-gold)
   * [6.2 Time-Based Train/Test Split](#6-2-split)
   * [6.3 Train & Compare Models](#6-3-models)
   * [6.4 Accuracy by Meter & Feature Importance](#6-4-importance)
7. [Anomaly Detection](#7-anomalies)
   * [7.1 Flag Unusual Readings](#7-1-flag)
   * [7.2 Likely Causes & Review Sample](#7-2-causes)
   * [7.3 Buildings Using More Electricity Than Expected](#7-3-excess)
8. [Key Findings](#8-findings)

<a id="1-env-setup"></a>
## 1. Environment Setup & Library Imports

In [ ]:
# Ignore PyArrow timezone warnings
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

In [ ]:
# Import Spark, pandas, plotting, and utility libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnull
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, TimestampType
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import Window
from pyspark.sql import functions as F
from pandas.tseries.holiday import USFederalHolidayCalendar
from pyspark.pandas.utils import PandasAPIOnSparkAdviceWarning
import pyspark.pandas as ps
import numpy as np
import zipfile
import warnings

# Ignore FutureWarnings generated by the PySpark Pandas API
warnings.simplefilter(action='ignore', category=FutureWarning)
from pyspark.pandas.utils import PandasAPIOnSparkAdviceWarning

# Prevent pandas-on-spark from failing on ANSI mode and avoid heavy sequence indexing
ps.set_option("compute.fail_on_ansi_mode", False)
ps.set_option("compute.default_index_type", "distributed")

<a id="2-data-download"></a>
## 2. Dataset Download & Extraction
Once your [Kaggle API credentials](https://www.kaggle.com/docs/api) are configured, run the cell below to download and extract the [ASHRAE competition dataset](https://www.kaggle.com/competitions/ashrae-energy-prediction/data).

### Configure Kaggle Credentials via Colab Secrets

In [ ]:
import os
import json
from google.colab import userdata

# Retrieve your API token secrets securely from Colab Secrets
kaggle_user = userdata.get('KAGGLE_USERNAME').strip()
kaggle_key = userdata.get('KAGGLE_KEY').strip()

# Setup directory and write kaggle.json
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

token_path = os.path.join(kaggle_dir, 'kaggle.json')
with open(token_path, 'w') as f:
    json.dump({'username': kaggle_user, 'key': kaggle_key}, f)

os.chmod(token_path, 0o600)
print(f"Successfully configured kaggle.json for user: {kaggle_user}!")

In [ ]:
competition_name = 'ashrae-energy-prediction'
zip_path = f"{competition_name}.zip"

# Define the path to the data folder
extract_dir = 'data/'

if not os.path.exists(extract_dir) or len(os.listdir(extract_dir)) == 0:
    print("Downloading dataset...")
    !kaggle competitions download -c {competition_name}

    if os.path.exists(zip_path):
        print(f"Extracting files into {extract_dir}/...")
        os.makedirs(extract_dir, exist_ok=True)
        import zipfile
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        print("Files extracted successfully!")
    else:
        print("Download failed. Verify that your competition credentials have accepted the rules on the Kaggle competition page.")
else:
    print(f"Data folder '{extract_dir}' already exists and contains files.")

<a id="3-spark-loading"></a>
## 3. Spark Session & Data Loading
Initialize the `SparkSession`, load the raw CSV files into Spark DataFrames, and register temporary SQL views (`train`, `weather_train`, `building_metadata`).

In [ ]:
# 6 GB driver memory so the 20-million-row joins and the models fit (Spark's default is 1 GB; ignored on Databricks)
spark = SparkSession.builder \
    .appName("ASHRAE Energy Prediction") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()

In [ ]:
# Load the datasets into Spark DataFrames with headers and inferred schemas
building_metadata_df = spark.read.csv(f"{extract_dir}building_metadata.csv", header=True, inferSchema=True)
train_df = spark.read.csv(f"{extract_dir}train.csv", header=True, inferSchema=True)
weather_train_df = spark.read.csv(f"{extract_dir}weather_train.csv", header=True, inferSchema=True)

# Register DataFrames as temporary SQL views
train_df.createOrReplaceTempView("train")
weather_train_df.createOrReplaceTempView("weather_train")
building_metadata_df.createOrReplaceTempView("building_metadata")

In [ ]:
# Print the schema for each DataFrame
building_metadata_df.printSchema()
train_df.printSchema()
weather_train_df.printSchema()

In [ ]:
# Disable ANSI mode for pandas-on-Spark compatibility
spark.conf.set("spark.sql.ansi.enabled", "false")

<a id="4-cleaning-fe"></a>
## 4. Data Cleaning & Feature Engineering
* **Time features:** `hour_of_day`, `day_of_week` (Mon=0), `month`, `is_weekend`, `is_holiday` (US federal holidays), and `us_season`
* **Weather features:** `temp_threshold` and `air_temp_avg_24h` – rolling average air temperature over the last 24 hours, per site
* **Building features:** `building_age_group` (Historic / Older / Modern / New / Unknown) and `building_size` (small / medium / large / massive by square-feet quartile)

<a id="4-1-building-metadata"></a>
### 4.1 Building Metadata Cleaning & Age Cohorts
[↑ Back to Table of Contents](#toc)

In [ ]:
# Convert building_metadata to a pandas-on-Spark DataFrame
building_metadata = building_metadata_df.pandas_api()

In [ ]:
# Check for missing values per column
building_metadata.isnull().sum()

In [ ]:
# Flag missing values before filling (53% of buildings lack year_built, 75% lack floor_count),
# so charts and models can tell recorded values from filled ones
building_metadata['year_built_missing'] = building_metadata['year_built'].isnull().astype(int)
building_metadata['floor_count_missing'] = building_metadata['floor_count'].isnull().astype(int)

# Fill floor_count with a logical default of 1
building_metadata['floor_count'] = building_metadata['floor_count'].fillna(1)

# Fill year_built with the median year
median_year = building_metadata['year_built'].median()
building_metadata['year_built'] = building_metadata['year_built'].fillna(median_year)

In [ ]:
# Verify no missing values remain in building_metadata
building_metadata.isnull().sum()

In [ ]:
# 1. Bring building_metadata to local pandas
bm = building_metadata.to_pandas()

# 2. Define the bins and labels for the 4 categories
age_bins = [-np.inf, 1959, 1979, 1999, np.inf]
age_labels = [
    'Historic', # < 1960
    'Older',    # 1960 - 1979
    'Modern',   # 1980 - 1999
    'New'       # 2000+
]

# 3. Create the feature
bm["building_age_group"] = pd.cut(
    bm["year_built"],
    bins=age_bins,
    labels=age_labels
).astype(str)

# Buildings with no recorded year_built get their own group instead of the median year's group
bm.loc[bm["year_built_missing"] == 1, "building_age_group"] = "Unknown"

# 4. Push back to Spark and register the updated view
building_metadata = ps.from_pandas(bm)
building_metadata.to_spark(index_col="index").createOrReplaceTempView("building_metadata")

<a id="4-2-train-data"></a>
### 4.2 Meter Data Cleaning
[↑ Back to Table of Contents](#toc)

The meter readings have known data-quality problems that distort every average if left in:
* **Building 1099 steam (meter 2)** reports impossibly large values that dominate all steam totals.
* **Site 0 electricity** is zero until 20 May 2016 (meters not yet online) and is recorded in kBTU, not kWh (1 kBTU = 0.2931 kWh).
* **Zero electricity readings** mean a sensor or logging failure, since an occupied building always draws some power. Zeros on other meters can be real (for example, no heating in summer), so they are kept.

In [ ]:
# Attach site_id so site-specific rules can be applied
t = train_df.join(building_metadata_df.select("building_id", "site_id"), on="building_id", how="left")
rows_before = t.count()

train_clean = (t
    # 1. Remove the corrupted steam meter of building 1099
    .filter(~((F.col("building_id") == 1099) & (F.col("meter") == 2)))
    # 2. Remove site 0 electricity before its meters came online
    .filter(~((F.col("site_id") == 0) & (F.col("meter") == 0) &
              (F.col("timestamp") < F.lit("2016-05-21").cast("timestamp"))))
    # 3. Remove zero electricity readings (sensor/logging failures)
    .filter(~((F.col("meter") == 0) & (F.col("meter_reading") == 0)))
    # 4. Convert site 0 electricity from kBTU to kWh
    .withColumn("meter_reading",
        F.when((F.col("site_id") == 0) & (F.col("meter") == 0), F.col("meter_reading") * 0.2931)
         .otherwise(F.col("meter_reading")))
    .drop("site_id"))

rows_after = train_clean.count()
print(f"Rows before: {rows_before:,} | after: {rows_after:,} | removed: {rows_before - rows_after:,} "
      f"({(rows_before - rows_after) / rows_before:.1%})")

# Replace the train view with the cleaned data
train_clean.createOrReplaceTempView("train")
train = train_clean.pandas_api()

<a id="4-3-weather-data"></a>
### 4.3 Weather Data Cleaning & Time/Rolling Features
[↑ Back to Table of Contents](#toc)

In [ ]:
# Preview the weather_train table
weather = spark.sql("""
    SELECT *
    FROM weather_train
""")

In [ ]:
# Convert weather to a pandas-on-Spark DataFrame
weather = weather.pandas_api()

In [ ]:
# Check for missing values in the weather table
weather.isnull().sum()

In [ ]:
# Ensure data is sorted chronologically before forward-filling
weather = weather.sort_values(by=['site_id', 'timestamp'])

In [ ]:
# Convert 7MB weather table to pandas for ultra-fast, zero-overhead cleaning
w_df = weather.to_pandas()

# 0. Align time zones: weather timestamps are UTC, meter timestamps are local time.
#    Shift each site's weather to local time (hour offsets per site_id 0-15).
site_utc_offset = {0: -5, 1: 0, 2: -7, 3: -5, 4: -8, 5: 0, 6: -5, 7: -5,
                   8: -5, 9: -6, 10: -7, 11: -5, 12: 0, 13: -6, 14: -5, 15: -5}
w_df["timestamp"] = w_df["timestamp"] + pd.to_timedelta(w_df["site_id"].map(site_utc_offset), unit="h")
w_df = w_df.sort_values(["site_id", "timestamp"])

cols_ffill = ["air_temperature", "dew_temperature", "wind_speed", "wind_direction", "sea_level_pressure"]

# 1. Group by site_id and forward-fill, then backward-fill
w_df[cols_ffill] = w_df.groupby("site_id")[cols_ffill].ffill()
w_df[cols_ffill] = w_df.groupby("site_id")[cols_ffill].bfill()

# 2. Fill cloud coverage with median
w_df["cloud_coverage"] = w_df["cloud_coverage"].fillna(w_df["cloud_coverage"].median())

# 3. Fill missing precipitation values with 0
w_df["precip_depth_1_hr"] = w_df["precip_depth_1_hr"].fillna(0)

# 4. Fill remaining sea level pressure with global median
w_df["sea_level_pressure"] = w_df["sea_level_pressure"].fillna(w_df["sea_level_pressure"].median())

# Convert back to pandas-on-spark
weather = ps.from_pandas(w_df)

In [ ]:
# Verify no missing values remain in the weather table
weather.isnull().sum()

In [ ]:
# Get US federal holidays from 2015 to 2019
holidays = (USFederalHolidayCalendar()
            .holidays(start="2015-01-01", end="2019-12-31")
            .strftime("%Y-%m-%d")
            .tolist())

# Map month numbers to US seasons
season_mapping = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall', 10: 'Fall', 11: 'Fall'
}

# Define a 24-hour rolling window (current hour + previous 23 hours) per site
w_24h = (Window.partitionBy("site_id")
         .orderBy(F.col("timestamp").cast("long"))
         .rangeBetween(-23 * 3600, 0))


# Function to extract time, weekend, holiday, and season features
def add_time_features(df):
    df["hour_of_day"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["month"] = df["timestamp"].dt.month
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["is_holiday"] = df["timestamp"].dt.strftime("%Y-%m-%d").isin(holidays).astype(int)
    df["us_season"] = df["month"].map(season_mapping)
    return df

# Function to categorize temperature thresholds and calculate 24-hour average temperature
def add_advanced_weather_features(df):
    return (df.to_spark()
              .withColumn("temp_threshold",
                  F.when(F.col("air_temperature") < 0, 'Freezing')
                  .when((F.col("air_temperature") >= 0) & (F.col("air_temperature") <= 10), 'Cold')
                  .when((F.col("air_temperature") > 10) & (F.col("air_temperature") <= 20), 'Comfort')
                  .when((F.col("air_temperature") > 20) & (F.col("air_temperature") <= 30), 'Warm')
                  .otherwise('Extreme Heat')
              )
              .withColumn("air_temp_avg_24h", F.avg("air_temperature").over(w_24h)))


# Apply time and weather feature functions
weather = add_time_features(weather)
weather = add_advanced_weather_features(weather)

# Register updated weather view and convert back to pandas-on-Spark
weather.createOrReplaceTempView("weather")
weather = weather.pandas_api()

<a id="4-4-building-features"></a>
### 4.4 Building Size Quartiles & Schema Verification
[↑ Back to Table of Contents](#toc)

In [ ]:
# Categorize buildings into size quartiles
bm = building_metadata.to_pandas()
bm["building_size"] = pd.qcut(bm["square_feet"], q=4,
                              labels=["small", "medium", "large", "massive"]).astype(str)
building_metadata = ps.from_pandas(bm)

# 2. Add index_col="index" to resolve the missing index warning
building_metadata.to_spark(index_col="index").createOrReplaceTempView("building_metadata")

In [ ]:
# Print updated schemas to verify new features
building_metadata.to_spark().printSchema()
train.to_spark().printSchema()
weather.to_spark().printSchema()

<a id="5-business-questions"></a>
## 5. Business Questions & Spark SQL Analysis
[↑ Back to Table of Contents](#toc)

<a id="5-1-daily-cycles"></a>
### 5.1 Daily Operational Cycles
To analyze daily usage patterns, this query joins the cleaned meter data with the engineered `hour_of_day` feature and averages `meter_reading` **per meter type**. Meters use different units, so each one is shown as an index (100 = that meter's average hour) so they can share one chart:

In [ ]:
# Calculate average meter reading for each hour of the day, per meter type
result = spark.sql("""
SELECT /*+ BROADCAST(b), BROADCAST(w) */
    w.hour_of_day,
    t.meter,
    AVG(t.meter_reading) AS avg_hourly_usage
FROM
    train t
JOIN
    building_metadata b ON t.building_id = b.building_id
JOIN
    weather w ON b.site_id = w.site_id AND t.timestamp = w.timestamp
GROUP BY
    w.hour_of_day,
    t.meter
ORDER BY
    t.meter,
    w.hour_of_day
""")

# Display the hourly usage results (24 hours x 4 meters)
result.show(96)

In [ ]:
# Convert SQL result to pandas and index each meter to its own daily average (=100)
meter_names = {0: "Electricity", 1: "Chilled Water", 2: "Steam", 3: "Hot Water"}
df_hourly = result.toPandas()
df_hourly["meter_name"] = df_hourly["meter"].map(meter_names)
df_hourly_pivot = df_hourly.pivot(index="hour_of_day", columns="meter_name", values="avg_hourly_usage")
df_hourly_index = df_hourly_pivot / df_hourly_pivot.mean() * 100

# Plot one line per meter type
df_hourly_index.plot(figsize=(10, 5), marker="o", linewidth=2)
plt.title("Daily Operational Cycles: Hourly Usage by Meter Type (Index, 100 = Daily Average)")
plt.xlabel("Hour of Day (0 - 23, local time)")
plt.ylabel("Usage Index")
plt.xticks(range(0, 24))
plt.axhline(100, color="grey", linewidth=1)
plt.legend(title="Meter Type")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# Peak-to-trough swing per meter
print((df_hourly_index.max() - df_hourly_index.min()).round(1).rename("peak-to-trough swing (index points)"))

<a id="5-2-weekly-drops"></a>
### 5.2 Weekly Usage Drops
To compare weekday versus weekend usage, this query uses the engineered `day_of_week` (Mon=0) and `is_weekend` features and aggregates `meter_reading` per meter type:

In [ ]:
# Calculate average meter reading for each day of the week, per meter type
result = spark.sql("""
SELECT /*+ BROADCAST(b), BROADCAST(w) */
    w.day_of_week,
    t.meter,
    AVG(t.meter_reading) AS avg_daily_usage
FROM
    train t
JOIN
    building_metadata b ON t.building_id = b.building_id
JOIN
    weather w ON b.site_id = w.site_id AND t.timestamp = w.timestamp
GROUP BY
    w.day_of_week,
    t.meter
ORDER BY
    t.meter,
    w.day_of_week
""")

# Display the daily usage results (7 days x 4 meters)
result.show(28)

In [ ]:
# Convert SQL result to pandas, map day numbers (Mon=0) to names, and index each meter to its weekly average
df_weekly = result.toPandas()
day_labels = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}
df_weekly["day_name"] = df_weekly["day_of_week"].map(day_labels)
df_weekly["meter_name"] = df_weekly["meter"].map(meter_names)
df_weekly_pivot = (df_weekly.pivot(index="day_name", columns="meter_name", values="avg_daily_usage")
                   .reindex(list(day_labels.values())))
df_weekly_index = df_weekly_pivot / df_weekly_pivot.mean() * 100

# Plot grouped bars of usage index by day of week
df_weekly_index.plot(kind="bar", figsize=(10, 5), edgecolor="black")
plt.title("Weekly Usage Drops: Usage by Day of Week and Meter Type (Index, 100 = Weekly Average)")
plt.xlabel("Day of Week")
plt.ylabel("Usage Index")
plt.xticks(rotation=0)
plt.axhline(100, color="grey", linewidth=1)
plt.legend(title="Meter Type")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# Weekend drop vs weekday average, per meter
weekday = df_weekly_pivot.loc[["Mon", "Tue", "Wed", "Thu", "Fri"]].mean()
weekend = df_weekly_pivot.loc[["Sat", "Sun"]].mean()
print(((weekend / weekday - 1) * 100).round(1).rename("weekend vs weekday (%)"))

<a id="5-3-seasonal-switching"></a>
### 5.3 Seasonal Fuel Switching
To observe how different utility types fluctuate across seasons, this query joins `train`, `building_metadata`, and `weather` to group total `meter_reading` by US season (`us_season`) and `meter` type:

In [ ]:
# Join train, building_metadata, and weather to calculate total seasonal usage by US season and meter type
result = spark.sql("""
SELECT /*+ BROADCAST(b), BROADCAST(w) */
    w.us_season,
    t.meter,
    ROUND(SUM(t.meter_reading), 2) AS total_seasonal_usage
FROM
    train t
JOIN
    building_metadata b ON t.building_id = b.building_id
JOIN
    weather w ON b.site_id = w.site_id AND t.timestamp = w.timestamp
GROUP BY
    w.us_season,
    t.meter
ORDER BY
    CASE w.us_season
        WHEN 'Winter' THEN 1
        WHEN 'Spring' THEN 2
        WHEN 'Summer' THEN 3
        WHEN 'Fall' THEN 4
    END,
    t.meter;
""")

# Display the seasonal fuel switching results
result.show(20, truncate=False)

In [ ]:
# Convert SQL result to pandas DataFrame and map meter IDs to names
df_seasonal = result.toPandas()
meter_names = {0: "Electricity", 1: "Chilled Water", 2: "Steam", 3: "Hot Water"}
df_seasonal["meter_name"] = df_seasonal["meter"].map(meter_names)

# Order seasons logically: Winter -> Spring -> Summer -> Fall
season_order = ["Winter", "Spring", "Summer", "Fall"]
df_seasonal_pivot = df_seasonal.pivot(index="us_season", columns="meter_name", values="total_seasonal_usage").reindex(season_order)

# Plot grouped bar chart of seasonal utility consumption
df_seasonal_pivot.plot(kind="bar", figsize=(10, 5), edgecolor="black")
plt.title("Seasonal Fuel Switching: Total Energy Usage by US Season and Meter Type")
plt.xlabel("US Season")
plt.ylabel("Total Usage (Log Scale)")
plt.yscale("log")
plt.xticks(rotation=0)
plt.legend(title="Meter Type")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

<a id="5-4-temp-thresholds"></a>
### 5.4 Temperature Thresholds
To identify how energy consumption responds when outdoor temperatures cross critical heating and cooling thresholds, this query joins `train`, `building_metadata`, and `weather` to aggregate usage across temperature bands and meter types:

In [ ]:
# Join train, building_metadata, and weather to calculate average usage by temperature threshold and meter
result = spark.sql("""
SELECT /*+ BROADCAST(b), BROADCAST(w) */
    w.temp_threshold,
    t.meter,
    COUNT(t.meter_reading) AS reading_count,
    ROUND(AVG(t.meter_reading), 2) AS avg_meter_reading
FROM
    train t
JOIN
    building_metadata b ON t.building_id = b.building_id
JOIN
    weather w ON b.site_id = w.site_id AND t.timestamp = w.timestamp
GROUP BY
    w.temp_threshold,
    t.meter
ORDER BY
    w.temp_threshold,
    t.meter;
""")

# Display the temperature threshold results
result.show(20, truncate=False)

In [ ]:
# Convert SQL result to pandas DataFrame and map meter names
df_temp = result.toPandas()
meter_names = {0: "Electricity", 1: "Chilled Water", 2: "Steam", 3: "Hot Water"}
df_temp["meter_name"] = df_temp["meter"].map(meter_names)

# Order temperature thresholds logically from Freezing to Extreme Heat
threshold_order = ["Freezing", "Cold", "Comfort", "Warm", "Extreme Heat"]
df_temp_pivot = df_temp.pivot(index="temp_threshold", columns="meter_name", values="avg_meter_reading").reindex(threshold_order)

# Plot grouped bar chart of average usage across temperature thresholds
df_temp_pivot.plot(kind="bar", figsize=(10, 5), edgecolor="black")
plt.title("Temperature Thresholds: Average Meter Reading by Temperature Band")
plt.xlabel("Outdoor Temperature Threshold")
plt.ylabel("Average Meter Reading (Log Scale)")
plt.yscale("log")
plt.xticks(rotation=0)
plt.legend(title="Meter Type")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

<a id="5-5-age-efficiency"></a>
### 5.5 Weather & Building Age Efficiency
To evaluate how building age impacts energy efficiency (usage per square foot) under varying weather conditions and seasons, this query groups buildings by construction era (`building_age_group`) and season (`us_season`). It uses **electricity only (meter 0)**, the one meter measured in kWh, so that usage per square foot is a true kWh/sqft figure. Buildings with no recorded construction year appear as their own **Unknown** group rather than being placed in the median year's group.

[↑ Back to Table of Contents](#toc)

In [ ]:
# Join tables to compare average usage and energy per square foot by building age group and season
result = spark.sql("""
SELECT /*+ BROADCAST(b), BROADCAST(w) */
    b.building_age_group,
    w.us_season,
    ROUND(AVG(w.air_temp_avg_24h), 2) AS avg_24h_temp,
    ROUND(AVG(t.meter_reading), 2) AS avg_meter_reading,
    ROUND(AVG(t.meter_reading / b.square_feet), 4) AS avg_kwh_per_sqft
FROM
    train t
JOIN
    building_metadata b ON t.building_id = b.building_id
JOIN
    weather w ON b.site_id = w.site_id AND t.timestamp = w.timestamp
WHERE
    t.meter = 0
GROUP BY
    b.building_age_group,
    w.us_season
ORDER BY
    b.building_age_group,
    w.us_season;
""")

In [ ]:
# Display the building age efficiency results
result.show(20, truncate=False)

# Convert SQL result to pandas DataFrame for plotting
df_age = result.toPandas()
age_order = ["Historic", "Older", "Modern", "New", "Unknown"]
season_order = ["Winter", "Spring", "Summer", "Fall"]

# Pivot data to compare energy intensity per sqft across building age groups and seasons
df_age_pivot = df_age.pivot(index="building_age_group", columns="us_season", values="avg_kwh_per_sqft").reindex(index=age_order, columns=season_order)

# Plot grouped bar chart of energy efficiency by building age group and season
df_age_pivot.plot(kind="bar", figsize=(10, 5), edgecolor="black")
plt.title("Weather & Building Age Efficiency: Average Electricity per Sqft by Era and Season")
plt.xlabel("Building Age Group")
plt.ylabel("Average Hourly Electricity per Square Foot (kWh / sqft)")
plt.xticks(rotation=0)
plt.legend(title="US Season")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

<a id="5-6-eui"></a>
### 5.6 Energy Use Intensity by Primary Use & Building Size
Research question 2 asks which building uses consume the most energy per square foot. **Energy Use Intensity (EUI)** is a building's yearly electricity divided by its floor area (kWh per sq ft per year). It is calculated per building first, then summarized with the median so a few extreme buildings do not dominate. Electricity is used because it is the one meter recorded in kWh.

[↑ Back to Table of Contents](#toc)

In [ ]:
# Electricity EUI per building: average hourly kWh x 8,784 hours (2016 had 366 days) / floor area
spark.sql("""
SELECT
    b.building_id,
    b.primary_use,
    b.building_size,
    AVG(t.meter_reading) * 8784 / b.square_feet AS kwh_per_sqft_year
FROM
    train t
JOIN
    building_metadata b ON t.building_id = b.building_id
WHERE
    t.meter = 0
GROUP BY
    b.building_id,
    b.primary_use,
    b.building_size,
    b.square_feet
""").createOrReplaceTempView("building_eui")

# Median EUI by primary use
result = spark.sql("""
SELECT
    primary_use,
    COUNT(*) AS buildings,
    ROUND(PERCENTILE_APPROX(kwh_per_sqft_year, 0.5), 2) AS median_kwh_per_sqft_year,
    ROUND(AVG(kwh_per_sqft_year), 2) AS mean_kwh_per_sqft_year
FROM
    building_eui
GROUP BY
    primary_use
ORDER BY
    median_kwh_per_sqft_year DESC
""")

result.show(20, truncate=False)

In [ ]:
# Plot median EUI by primary use, with the number of buildings behind each bar
df_eui = result.toPandas().sort_values("median_kwh_per_sqft_year")

plt.figure(figsize=(10, 6))
bars = plt.barh(df_eui["primary_use"], df_eui["median_kwh_per_sqft_year"], color="darkorange", edgecolor="black")
for bar, n in zip(bars, df_eui["buildings"]):
    plt.text(bar.get_width(), bar.get_y() + bar.get_height() / 2, f"  n={n}", va="center", fontsize=9)
plt.title("Electricity Use Intensity by Primary Use (Median per Building, 2016)")
plt.xlabel("Median Electricity Use (kWh per sq ft per year)")
plt.ylabel("Primary Use")
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Median EUI by building size quartile
result_size = spark.sql("""
SELECT
    building_size,
    COUNT(*) AS buildings,
    ROUND(PERCENTILE_APPROX(kwh_per_sqft_year, 0.5), 2) AS median_kwh_per_sqft_year
FROM
    building_eui
GROUP BY
    building_size
ORDER BY
    CASE building_size
        WHEN 'small' THEN 1
        WHEN 'medium' THEN 2
        WHEN 'large' THEN 3
        WHEN 'massive' THEN 4
    END
""")
result_size.show()

# Plot median EUI by size quartile
df_size = result_size.toPandas()
plt.figure(figsize=(7, 4))
plt.bar(df_size["building_size"], df_size["median_kwh_per_sqft_year"], color="goldenrod", edgecolor="black")
plt.title("Electricity Use Intensity by Building Size Quartile")
plt.xlabel("Building Size (square-feet quartile)")
plt.ylabel("Median kWh per sq ft per year")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

<a id="5-7-temp-corr"></a>
### 5.7 Temperature vs Usage
Research question 1 also asks how strongly outdoor temperature drives usage. Two views:
* **Scatter:** average reading at each whole degree of outdoor temperature, one panel per meter type (dot size = number of readings).
* **Correlation:** the correlation between temperature and usage inside each building-meter, summarized as the median per meter type. Positive values mean usage rises with heat (cooling); negative values mean usage rises with cold (heating).

[↑ Back to Table of Contents](#toc)

In [ ]:
# Average reading at each whole degree Celsius, per meter type
result = spark.sql("""
SELECT /*+ BROADCAST(b), BROADCAST(w) */
    t.meter,
    ROUND(w.air_temperature) AS temp_c,
    COUNT(*) AS readings,
    AVG(t.meter_reading) AS avg_reading
FROM
    train t
JOIN
    building_metadata b ON t.building_id = b.building_id
JOIN
    weather w ON b.site_id = w.site_id AND t.timestamp = w.timestamp
GROUP BY
    t.meter,
    ROUND(w.air_temperature)
HAVING
    COUNT(*) >= 1000
ORDER BY
    t.meter,
    temp_c
""")
df_tb = result.toPandas()

# One scatter panel per meter type, since each meter has its own units
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (m, name) in zip(axes.flat, meter_names.items()):
    d = df_tb[df_tb["meter"] == m]
    ax.scatter(d["temp_c"], d["avg_reading"], s=np.clip(d["readings"] / d["readings"].max() * 200, 10, 200),
               alpha=0.7, color="steelblue", edgecolor="black")
    ax.set_title(name)
    ax.set_xlabel("Outdoor Air Temperature (°C)")
    ax.set_ylabel("Average Meter Reading")
    ax.grid(True, linestyle="--", alpha=0.6)
fig.suptitle("Temperature vs Usage by Meter Type (dot size = number of readings)")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation between temperature and usage inside each building-meter, summarized per meter type
pairs = spark.sql("""
SELECT /*+ BROADCAST(b), BROADCAST(w) */
    t.building_id,
    t.meter,
    t.meter_reading,
    w.air_temperature
FROM
    train t
JOIN
    building_metadata b ON t.building_id = b.building_id
JOIN
    weather w ON b.site_id = w.site_id AND t.timestamp = w.timestamp
""")

corr_by_meter = (pairs.groupBy("building_id", "meter")
                 .agg(F.corr("air_temperature", "meter_reading").alias("r"))
                 .filter(F.col("r").isNotNull() & ~F.isnan("r"))
                 .groupBy("meter")
                 .agg(F.count("r").alias("building_meters"),
                      F.round(F.expr("percentile_approx(r, 0.5)"), 2).alias("median_correlation"),
                      F.round(F.avg((F.col("r") > 0).cast("int")), 2).alias("share_positive"))
                 .orderBy("meter")
                 .toPandas())
corr_by_meter["meter_name"] = corr_by_meter["meter"].map(meter_names)
corr_by_meter[["meter_name", "building_meters", "median_correlation", "share_positive"]]

<a id="6-modeling"></a>
## 6. Gold Feature Table & Prediction Models
Research question 3: how accurately can hourly energy use be predicted for October–December, and which features matter most?

<a id="6-1-gold"></a>
### 6.1 Gold Feature Table
The cleaned meter readings are joined with building and weather features into one model-ready **gold** table and saved as Parquet, so the models do not repeat the joins. On Databricks, save it as a Delta table instead: `gold.write.format("delta").mode("overwrite").saveAsTable("gold_energy_features")`.

The target is `label = log1p(meter_reading)`. Readings range from 0 to millions, so predicting the log keeps the largest buildings from dominating, and the error on this scale is **RMSLE**, the project's main metric.

[↑ Back to Table of Contents](#toc)

In [ ]:
# Build the gold feature table: one row per building, meter and hour
gold = spark.sql("""
SELECT /*+ BROADCAST(b), BROADCAST(w) */
    t.building_id,
    t.meter,
    t.timestamp,
    t.meter_reading,
    LOG1P(t.meter_reading) AS label,
    b.site_id,
    b.primary_use,
    LN(b.square_feet) AS log_sqft,
    b.year_built,
    b.year_built_missing,
    b.floor_count,
    b.floor_count_missing,
    w.air_temperature,
    w.dew_temperature,
    w.cloud_coverage,
    w.precip_depth_1_hr,
    w.sea_level_pressure,
    w.wind_speed,
    w.wind_direction,
    w.air_temp_avg_24h,
    w.hour_of_day,
    w.day_of_week,
    w.is_weekend,
    w.is_holiday
FROM
    train t
JOIN
    building_metadata b ON t.building_id = b.building_id
JOIN
    weather w ON b.site_id = w.site_id AND t.timestamp = w.timestamp
""")

# Save once, then read back so later steps start from the saved table
gold_path = "gold/energy_features"
gold.write.mode("overwrite").parquet(gold_path)
gold = spark.read.parquet(gold_path)

print(f"Gold table rows: {gold.count():,}")
gold.printSchema()

<a id="6-2-split"></a>
### 6.2 Time-Based Train/Test Split
Models train on **January–September** and are tested on **October–December**, so they never see the period they are asked to predict. `month` and `us_season` are left out of the features because the test months never appear in training; temperature carries the seasonal signal instead.

Spark models on Colab's 2 CPU cores are slow on ~15 million rows, so the models train on a random sample of the training months. Raise `TRAIN_FRACTION` on Databricks for better accuracy.

[↑ Back to Table of Contents](#toc)

In [ ]:
TRAIN_FRACTION = 0.05   # share of Jan-Sep rows used for training (raise on Databricks)
TEST_FRACTION = 0.10    # share of Oct-Dec rows used to compare the models
split_date = F.lit("2016-10-01").cast("timestamp")

train_set = gold.filter(F.col("timestamp") < split_date).sample(fraction=TRAIN_FRACTION, seed=42).cache()
test_full = gold.filter(F.col("timestamp") >= split_date)
test_set = test_full.sample(fraction=TEST_FRACTION, seed=42).cache()

print(f"Training rows: {train_set.count():,} | Test rows (comparison sample): {test_set.count():,} | "
      f"Full Oct-Dec rows: {test_full.count():,}")

<a id="6-3-models"></a>
### 6.3 Train & Compare Models
Three models, from simple to complex:
* **Linear Regression (baseline):** meter, primary use, site, hour and weekday are one-hot encoded so the baseline can still learn a daily and weekly shape.
* **Random Forest:** many decision trees averaged together; captures non-linear effects such as heating below and cooling above a comfort band.
* **Gradient Boosting (Spark `GBTRegressor`):** trees built one after another, each correcting the errors of the last. It uses the same boosting method as XGBoost and LightGBM and runs in plain PySpark, so the notebook works on Colab and Databricks with no extra installs.

Metrics are on the log scale: **RMSLE** (main metric, lower is better), **MAE** (average absolute log error, lower is better) and **R²** (share of variation explained, higher is better). Expect training to take 20–40 minutes in total on Colab.

[↑ Back to Table of Contents](#toc)

In [ ]:
import time
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Numeric inputs shared by all models
numeric_features = ["log_sqft", "year_built", "year_built_missing", "floor_count", "floor_count_missing",
                    "air_temperature", "dew_temperature", "cloud_coverage", "precip_depth_1_hr",
                    "sea_level_pressure", "wind_speed", "wind_direction", "air_temp_avg_24h",
                    "is_weekend", "is_holiday"]

# Tree models use raw values: they can split on IDs, categories and time slots directly
tree_features = ["building_id", "site_id", "meter", "primary_use_idx", "hour_of_day", "day_of_week"] + numeric_features

# Turn primary_use text into a number
indexer = StringIndexer(inputCol="primary_use", outputCol="primary_use_idx", handleInvalid="keep")

# Linear baseline: one-hot encode categories and time slots so it can learn a daily and weekly shape
encoder = OneHotEncoder(inputCols=["meter", "primary_use_idx", "site_id", "hour_of_day", "day_of_week"],
                        outputCols=["meter_ohe", "use_ohe", "site_ohe", "hour_ohe", "dow_ohe"],
                        handleInvalid="keep")
assembler_lr = VectorAssembler(inputCols=["meter_ohe", "use_ohe", "site_ohe", "hour_ohe", "dow_ohe"] + numeric_features,
                               outputCol="features", handleInvalid="skip")
assembler_tree = VectorAssembler(inputCols=tree_features, outputCol="features", handleInvalid="skip")

pipelines = {
    "Linear Regression": Pipeline(stages=[
        indexer, encoder, assembler_lr,
        LinearRegression(featuresCol="features", labelCol="label")]),
    "Random Forest": Pipeline(stages=[
        indexer, assembler_tree,
        RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=30, maxDepth=12,
                              maxBins=128, subsamplingRate=0.7, maxMemoryInMB=512, seed=42)]),
    "Gradient Boosting": Pipeline(stages=[
        indexer, assembler_tree,
        GBTRegressor(featuresCol="features", labelCol="label", maxIter=60, maxDepth=6, stepSize=0.15,
                     maxBins=128, subsamplingRate=0.8, maxMemoryInMB=512, seed=42)]),
}

In [ ]:
def score(model, df):
    """Predict log1p(reading) and clip at 0, since a reading cannot be negative."""
    return model.transform(df).withColumn("prediction", F.greatest(F.col("prediction"), F.lit(0.0)))

evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction")
fitted, rows = {}, []

# Train each model on Jan-Sep and score it on the Oct-Dec sample
for name, pipe in pipelines.items():
    start = time.time()
    fitted[name] = pipe.fit(train_set)
    minutes = (time.time() - start) / 60
    preds = score(fitted[name], test_set).cache()
    rows.append({"Model": name,
                 "RMSLE": evaluator.evaluate(preds, {evaluator.metricName: "rmse"}),
                 "MAE (log)": evaluator.evaluate(preds, {evaluator.metricName: "mae"}),
                 "R² (log)": evaluator.evaluate(preds, {evaluator.metricName: "r2"}),
                 "Train minutes": round(minutes, 1)})
    preds.unpersist()
    print(f"{name}: trained in {minutes:.1f} min")

results_df = pd.DataFrame(rows).sort_values("RMSLE").reset_index(drop=True).round(4)
results_df

In [ ]:
# Compare models on RMSLE (best model highlighted)
colors = ["seagreen"] + ["grey"] * (len(results_df) - 1)
plt.figure(figsize=(8, 4))
plt.bar(results_df["Model"], results_df["RMSLE"], color=colors, edgecolor="black")
for i, v in enumerate(results_df["RMSLE"]):
    plt.text(i, v, f"{v:.3f}", ha="center", va="bottom")
plt.title("Model Comparison on October–December 2016 (lower RMSLE is better)")
plt.ylabel("RMSLE")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

baseline_rmsle = results_df.loc[results_df["Model"] == "Linear Regression", "RMSLE"].iloc[0]
improvement = (1 - results_df["RMSLE"].iloc[0] / baseline_rmsle) * 100
print(f"Best model: {results_df['Model'].iloc[0]} | RMSLE {improvement:.1f}% lower than the linear baseline")

<a id="6-4-importance"></a>
### 6.4 Accuracy by Meter & Feature Importance
The proposal expects electricity to be the most predictable meter and hot water the least. This breaks the best model's RMSLE down by meter type and shows which features the best tree model relied on most.

[↑ Back to Table of Contents](#toc)

In [ ]:
best_name = results_df["Model"].iloc[0]
best_model = fitted[best_name]
best_tree_name = [n for n in results_df["Model"] if n != "Linear Regression"][0]
print(f"Best model by RMSLE: {best_name}")

# RMSLE per meter type for the best model
rmsle_by_meter = (score(best_model, test_set)
                  .groupBy("meter")
                  .agg(F.sqrt(F.avg(F.pow(F.col("prediction") - F.col("label"), 2))).alias("RMSLE"),
                       F.count("*").alias("test_rows"))
                  .orderBy("RMSLE")
                  .toPandas())
rmsle_by_meter["meter_name"] = rmsle_by_meter["meter"].map(meter_names)
print(rmsle_by_meter[["meter_name", "RMSLE", "test_rows"]].round(4).to_string(index=False))

# Feature importance from the best tree model
importances = fitted[best_tree_name].stages[-1].featureImportances.toArray()
df_imp = pd.DataFrame({"feature": tree_features, "importance": importances}).sort_values("importance")

plt.figure(figsize=(9, 6))
plt.barh(df_imp["feature"], df_imp["importance"], color="slateblue", edgecolor="black")
plt.title(f"Feature Importance ({best_tree_name})")
plt.xlabel("Share of Total Importance")
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

<a id="7-anomalies"></a>
## 7. Anomaly Detection
Research question 4: what do the largest prediction errors reveal? The best model predicts what each building *should* use every hour from October to December. A reading is flagged when its error (residual) is far outside that building-meter's normal error range: more than 3 standard deviations from its average error (|z| ≥ 3). This works like a control chart on the model's residuals.

<a id="7-1-flag"></a>
### 7.1 Flag Unusual Readings
[↑ Back to Table of Contents](#toc)

In [ ]:
Z_THRESHOLD = 3.0

bm_window = Window.partitionBy("building_id", "meter")
time_window = Window.partitionBy("building_id", "meter").orderBy("timestamp")

# Score every Oct-Dec reading with the best model and standardize residuals per building-meter
scored = (score(best_model, test_full)
          .withColumn("expected_reading", F.expm1(F.col("prediction")))
          .withColumn("residual", F.col("label") - F.col("prediction"))
          .withColumn("z_score", (F.col("residual") - F.avg("residual").over(bm_window))
                                 / F.stddev("residual").over(bm_window))
          .withColumn("prev_reading", F.lag("meter_reading").over(time_window))
          .withColumn("is_anomaly", F.coalesce(F.abs(F.col("z_score")) >= Z_THRESHOLD, F.lit(False)))
          .select("building_id", "meter", "site_id", "primary_use", "timestamp", "hour_of_day",
                  "is_weekend", "is_holiday", "meter_reading", "prev_reading", "expected_reading",
                  "residual", "z_score", "is_anomaly"))

# Save the scored readings so the window calculations run only once
scored.write.mode("overwrite").parquet("gold/scored_oct_dec")
scored = spark.read.parquet("gold/scored_oct_dec")

total = scored.count()
flagged = scored.filter("is_anomaly").count()
print(f"Readings scored: {total:,} | flagged as anomalies: {flagged:,} ({flagged / total:.2%})")

<a id="7-2-causes"></a>
### 7.2 Likely Causes & Review Sample
Each flagged reading gets a likely cause from simple rules based on the anomaly types in the proposal:

| Rule | Likely cause |
|---|---|
| Reading below expected **and** zero or unchanged from the previous hour | Meter fault (zero or flat reading) |
| Reading above expected **and** more than double the previous hour | Sudden spike (possible equipment problem) |
| Reading above expected at night (22:00–06:00), on a weekend or on a holiday | After-hours use (possible waste) |
| Any other high or low reading | Higher / lower than expected |

These are first guesses to guide inspection, not confirmed causes.

[↑ Back to Table of Contents](#toc)

In [ ]:
# Label each flagged reading with its most likely cause
anomalies = (scored.filter("is_anomaly")
    .withColumn("likely_cause",
        F.when((F.col("residual") < 0) &
               ((F.col("meter_reading") == 0) | (F.col("meter_reading") == F.col("prev_reading"))),
               "Meter fault (zero or flat reading)")
         .when((F.col("residual") > 0) & (F.col("meter_reading") > 2 * F.col("prev_reading")),
               "Sudden spike")
         .when((F.col("residual") > 0) &
               ((F.col("hour_of_day") < 6) | (F.col("hour_of_day") >= 22) |
                (F.col("is_weekend") == 1) | (F.col("is_holiday") == 1)),
               "After-hours use (possible waste)")
         .when(F.col("residual") > 0, "Higher than expected")
         .otherwise("Lower than expected"))
    .cache())

# Count flagged readings by likely cause and meter type
cause_table = (anomalies.groupBy("likely_cause", "meter").count().toPandas()
               .pivot(index="likely_cause", columns="meter", values="count")
               .fillna(0).astype(int)
               .rename(columns=meter_names))
cause_table["Total"] = cause_table.sum(axis=1)
cause_table = cause_table.sort_values("Total")
print(cause_table.sort_values("Total", ascending=False))

cause_table.drop(columns="Total").plot(kind="barh", stacked=True, figsize=(10, 5), edgecolor="black")
plt.title("Flagged Readings by Likely Cause and Meter Type (Oct–Dec 2016)")
plt.xlabel("Number of Flagged Readings")
plt.ylabel("Likely Cause")
plt.legend(title="Meter Type")
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Building-meters with the most flagged hours: the inspection list for facility managers
top_offenders = (anomalies.groupBy("building_id", "meter", "primary_use")
                 .agg(F.count("*").alias("flagged_hours"),
                      F.expr("mode(likely_cause)").alias("most_common_cause"))
                 .orderBy(F.desc("flagged_hours"))
                 .limit(10)
                 .toPandas())
top_offenders["meter"] = top_offenders["meter"].map(meter_names)
top_offenders

In [ ]:
# Weekly anomaly report: flagged readings and buildings affected per week
weekly = (anomalies.withColumn("week_start", F.date_trunc("week", F.col("timestamp")))
          .groupBy("week_start")
          .agg(F.count("*").alias("flagged_readings"),
               F.countDistinct("building_id").alias("buildings_affected"))
          .orderBy("week_start")
          .toPandas())

plt.figure(figsize=(10, 4))
plt.bar(weekly["week_start"].dt.strftime("%b %d"), weekly["buildings_affected"], color="indianred", edgecolor="black")
plt.title("Weekly Anomaly Report: Buildings with at Least One Flagged Reading")
plt.xlabel("Week Starting")
plt.ylabel("Buildings Affected")
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Random sample of flagged readings for manual review
review_sample = (anomalies.orderBy(F.rand(seed=42)).limit(15)
                 .select("building_id", "meter", "timestamp", "meter_reading", "prev_reading",
                         F.round("expected_reading", 2).alias("expected_reading"),
                         F.round("z_score", 2).alias("z_score"), "likely_cause")
                 .toPandas())
review_sample["meter"] = review_sample["meter"].map(meter_names)
review_sample

In [ ]:
# Example: actual vs expected for the building-meter with the most flagged hours
top = anomalies.groupBy("building_id", "meter").count().orderBy(F.desc("count")).first()
ex = (scored.filter((F.col("building_id") == top["building_id"]) & (F.col("meter") == top["meter"]))
      .select("timestamp", "meter_reading", "expected_reading", "is_anomaly")
      .orderBy("timestamp")
      .toPandas())
flag = ex[ex["is_anomaly"]]

plt.figure(figsize=(12, 5))
plt.plot(ex["timestamp"], ex["meter_reading"], linewidth=1, label="Actual reading")
plt.plot(ex["timestamp"], ex["expected_reading"], linewidth=1, alpha=0.8, label="Model expected")
plt.scatter(flag["timestamp"], flag["meter_reading"], color="red", s=15, zorder=3, label="Flagged anomaly")
plt.title(f"Building {top['building_id']} – {meter_names[top['meter']]}: Actual vs Expected (Oct–Dec 2016)")
plt.xlabel("Date")
plt.ylabel("Meter Reading")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

<a id="7-3-excess"></a>
### 7.3 Buildings Using More Electricity Than Expected
For owners and portfolio managers: total actual vs model-expected electricity per building for October–December. A large positive gap means a building used more than its own January–September behaviour predicts, which makes it a candidate for inspection or an efficiency upgrade. The same comparison, run after an upgrade, shows whether the upgrade actually saved energy.

[↑ Back to Table of Contents](#toc)

In [ ]:
# Actual vs expected electricity per building, Oct-Dec
excess = (scored.filter(F.col("meter") == 0)
          .groupBy("building_id", "primary_use")
          .agg(F.sum("meter_reading").alias("actual_kwh"),
               F.sum("expected_reading").alias("expected_kwh"))
          .filter(F.col("expected_kwh") > 10000)   # skip very small meters
          .withColumn("excess_kwh", F.col("actual_kwh") - F.col("expected_kwh"))
          .withColumn("excess_pct", F.round((F.col("actual_kwh") / F.col("expected_kwh") - 1) * 100, 1))
          .orderBy(F.desc("excess_pct")))

top_excess = excess.limit(10).toPandas()
top_excess[["actual_kwh", "expected_kwh", "excess_kwh"]] = top_excess[["actual_kwh", "expected_kwh", "excess_kwh"]].round(0)
print(top_excess.to_string(index=False))

plot_df = top_excess.sort_values("excess_pct")
plt.figure(figsize=(9, 5))
plt.barh("Bldg " + plot_df["building_id"].astype(str) + " (" + plot_df["primary_use"] + ")",
         plot_df["excess_pct"], color="firebrick", edgecolor="black")
plt.title("Top 10 Buildings Using More Electricity Than Expected (Oct–Dec 2016)")
plt.xlabel("Actual vs Expected Electricity (%)")
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

<a id="8-findings"></a>
## 8. Key Findings
Fill in after running the notebook top to bottom.

**RQ1 – Time and temperature patterns (5.1–5.4, 5.7)**
* Daily cycle: …
* Weekend drop: …
* Seasonal and temperature effect by meter: …

**RQ2 – Highest energy use per square foot (5.6)**
* …

**RQ3 – Prediction accuracy (6.3–6.4)**
* Best model and RMSLE vs the linear baseline: …
* Most and least predictable meter: …
* Top features: …

**RQ4 – What the largest errors reveal (7.1–7.3)**
* Share of readings flagged and the main likely causes: …
* Buildings to inspect first: …

**Recommendations**
* Facility managers: …
* Building owners and portfolio managers: …
* Finance teams: …

**Limitations:** one year of data (2016), so each season appears once; mostly university buildings; no occupancy, equipment or price data; time-zone offsets come from community analysis of the anonymized sites; US seasons and holidays are applied to every site, including the non-US ones; models train on a sample of the data.

[↑ Back to Table of Contents](#toc)